# DETR-based Predictive Railway Adhesion: training notebook

Code is ready but **nothing has been executed/trained**. Replace the synthetic dataset with real data (see README data contract) before training.

In [ ]:
import sys, torch
sys.path.insert(0, '..')
from adhesion import AdhesionDETR, ModelConfig, LossWeights, compute_loss
from adhesion.data import SyntheticRailDataset
from adhesion.train import fit, INPUT_KEYS
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

## 1. Configuration and model

In [ ]:
cfg = ModelConfig(image_size=(224, 384), n_axles=4, f_max_kn=100.0)
weights = LossWeights()
model = AdhesionDETR(cfg)
print(f'{sum(p.numel() for p in model.parameters())/1e6:.1f}M parameters')

## 2. Data
`SyntheticRailDataset` is a wiring placeholder. Swap in a `Dataset` returning the same keys (sequences of length T).

In [ ]:
train_ds = SyntheticRailDataset(cfg, n_samples=64, seq_len=4, seed=0)
val_ds = SyntheticRailDataset(cfg, n_samples=16, seq_len=4, seed=1)
{k: tuple(v.shape) for k, v in train_ds[0].items()}

## 3. Forward pass and loss check (single batch, no optimisation)

In [ ]:
batch = {k: torch.stack([train_ds[i][k] for i in range(2)]) for k in train_ds[0]}
model.eval()
with torch.no_grad():
    out = model(*[batch[k] for k in INPUT_KEYS], actuator_ok=batch['actuator_ok'])
    loss, logs = compute_loss(out, batch, cfg, weights)
print('Y_t = [F_cmd (kN), sigma_F (kN), a_t]')
print(out['f_cmd'].squeeze(-1), out['sigma_f'].squeeze(-1), out['valid'].squeeze(-1))
logs

## 4. Training
Set `RUN_TRAINING = True` to start training.

In [ ]:
RUN_TRAINING = False
if RUN_TRAINING:
    history = fit(model, train_ds, val_ds, epochs=20, batch_size=4, lr=1e-4,
                  weights=weights, device=device, ckpt_dir='../checkpoints')

## 5. Plot learning curves

In [ ]:
import matplotlib.pyplot as plt
if RUN_TRAINING:
    plt.plot([h['train']['total'] for h in history], label='train')
    plt.plot([h['val']['total'] for h in history], label='val')
    plt.xlabel('epoch'); plt.ylabel('total loss'); plt.legend(); plt.show()

## 6. Streaming inference
Carry the GRU state `h` across frames (T=1 per call).

In [ ]:
model.eval()
h = None
with torch.no_grad():
    for t in range(batch['images'].shape[1]):
        step = [batch[k][:, t:t+1] for k in INPUT_KEYS]
        out = model(*step, h_prev=h)
        h = out['h']
        print(t, out['f_cmd'].flatten().tolist())